In [1]:
import os
import sys
sys.path.append('..')

from src.vectorstore import csv_loader, build_vectorstore
from src.rag_pipeline import load_llm, semantic_retriever, initialize_rag_chain, format_docs_to_df
from src.hybrid import preprocess_query, bm25_retriever, hybrid_retriever
from src.prompts import prompt

c:\Users\liauw\Desktop\Sputnik\2025-26\Courses\block-6\575-nlp\DSCI_575_project_cliauwyt_cea\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Define model

In [3]:
llm = load_llm()

# RAG Semantic

## Load vector store

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

corpus_path = '../data/processed/preprocessed_corpus.csv'
vector_path = "../data/processed/vector_store"
docs = csv_loader(corpus_path)

if not os.path.exists(vector_path):
    build_vectorstore(docs, vector_path, embeddings)
    print(f"Saved vector store to {vector_path}")
    
vectorstore = FAISS.load_local(
    vector_path, embeddings, allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval

In [5]:
vector_retriever = semantic_retriever(vectorstore)
query = "what is the best soap"
format_docs_to_df(vector_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B0716PQVP2,Dealglad 10Pcs Double Layer Exfoliating Mesh S...,5.0,Must have with bars of soap ! You will love !
1,B08DV37PZV,"Palmolive Ultra Original Dish Liquid, 102 fl. ...",5.0,"That ""blue"" dish soap is more difficult to rin..."
2,B001HDZT7I,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,Although this had decent reviews when I resear...
3,B00O92P0KU,Lemon Essential Oil 4 Oz - 5x Extra Strength 1...,5.0,The Radha lemon essential oil is wonderful – a...
4,B08W2GY3KV,Beautywin Soft Silicone Bath Brush，Baby Shower...,1.0,Soap just falls right out it feel nice but ur ...


## Pipeline

In [6]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt)
print(rag_chain.invoke(query))

Based on the reviews above, it seems that there isn't a single best soap product mentioned. However, the reviews do provide some information about the types of soaps mentioned. 

One customer mentions that they like Palmolive because it cleans well and rinses off easily (Product ASIN: B08DV37PZV). This suggests that Palmolive is a good dish soap.

Another customer mentions that they make their own homemade dish soap using Lemongrass essential oil (Product ASIN: B00O92P0KU). The recipe involves using borax, grated bar soap, and essential oils, which indicates that a homemade soap made with natural ingredients is also a good option.

It's worth noting that the Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch (Product ASIN: B0716PQVP2) is not a soap product itself, but rather a soap saver pouch that can be used to hold and exfoliate soap bars.

Overall, the best soap seems to be a matter of personal preference, whether it's commercial (such as Palmolive) or homemade (using na

# Hybrid RAG: Semantic Search + BM25

### BM25 Retriever

In [7]:
keyword_retriever = bm25_retriever(docs)

# preprocess query so it's consistent with corpus preprocessing
query = preprocess_query(query)

# Retriever invoke returns the top_k docs
format_docs_to_df(keyword_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",4.0,A single sheet is good for a pair of underwear...
1,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,"If you MUST save on space, consider these laun..."
2,B00OPC61V6,Dial Mountain Fresh Antibacterial Deodorant So...,5.0,I use Dial soap because it does the best job o...
3,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,I bought this to keep with me on my recent tri...
4,B0773H89Y6,"- Soap Dispensing Dish Brush -,Multipurpose Ho...",1.0,The previous reviews were either really positi...


## Ensemble BM25 + Semantic Retriever

In [8]:
# Invoke to get combined results
ensemble_retriever = hybrid_retriever(keyword_retriever, vector_retriever)
format_docs_to_df(ensemble_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",4.0,A single sheet is good for a pair of underwear...
1,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,nice product
2,B004UJHY74,Sekkisei cream wash,5.0,Smells nice and soapy. Creamy. Does a great jo...
3,B0716PQVP2,Dealglad 10Pcs Double Layer Exfoliating Mesh S...,5.0,Must have with bars of soap ! You will love !
4,B08W2GY3KV,Beautywin Soft Silicone Bath Brush，Baby Shower...,1.0,Soap just falls right out it feel nice but ur ...
5,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,"If you MUST save on space, consider these laun..."
6,B00OPC61V6,Dial Mountain Fresh Antibacterial Deodorant So...,5.0,I use Dial soap because it does the best job o...
7,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,I bought this to keep with me on my recent tri...
8,B0773H89Y6,"- Soap Dispensing Dish Brush -,Multipurpose Ho...",1.0,The previous reviews were either really positi...


In [9]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt)
print(rag_chain.invoke(query))

Based on the provided reviews, it seems that a good soap is subjective and can depend on personal preferences. However, some popular choices mentioned in the reviews are:

1. **Sekkisei cream wash (B004UJHY74)**: This soap wash is described as "nice and soapy. Creamy. Does a great job" by a customer.

2. **Dial Mountain Fresh Antibacterial Deodorant Soap (B00OPC61V6)**: This soap is praised for fighting odor, leaving the user feeling clean and fresh, and having a nice fragrance.

3. **Travelon Hand Soap Toiletry Sheets (B001CGOPZM)**: While some customers had mixed opinions about this product, one customer stated that it "worked perfectly for my needs" and was safe to use in questionable water supplies.

It's essential to note that these are just a few examples and that the concept of "good soap" can vary greatly from person to person.
